# Experiment 4: Multi-Task Validation & Quantization

**Purpose:** Validate segmentation + pose claims (Section 10) and run QAT/PTQ (Section 5).

**Tasks:**
1. Segmentation — COCO-Seg
2. Pose estimation — COCO-Pose
3. PTQ — Post-Training Quantization
4. QAT — Quantization-Aware Training
5. INT8 model export

**Expected time:** ~6-8 hours total

In [ ]:
# === Cell 1: Setup ===
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q

import torch, os
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Part A: Segmentation Validation

In [ ]:
# === Cell 2: Train Segmentation — Quantized ===
os.system(
    "python scripts/train.py --task seg --variant quantized --data coco-val.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-q-416-seed42"
)

# === Train Segmentation — Standard ===
os.system(
    "python scripts/train.py --task seg --variant standard --data coco-val.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-std-416-seed42"
)
print("\n✅ Segmentation training complete!")

## Part B: Pose Estimation Validation

In [ ]:
# === Cell 3: Train Pose — Quantized ===
os.system(
    "python scripts/train.py --task pose --variant quantized --data coco8-pose.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-q-416-seed42"
)

# === Train Pose — Standard ===
os.system(
    "python scripts/train.py --task pose --variant standard --data coco8-pose.yaml "
    "--imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-std-416-seed42"
)
print("\n✅ Pose estimation training complete!")

In [ ]:
# === Cell 4: Multi-Task Results ===
import json, glob
from pathlib import Path

print("=" * 70)
print("  Multi-Task Validation Results")
print("=" * 70)

for pattern in ['seg-*-416-seed42', 'pose-*-416-seed42']:
    for f in sorted(glob.glob(f'experiments/results/{pattern}/config.json')):
        with open(f) as fh:
            cfg = json.load(fh)
        name = Path(f).parent.name
        fm = cfg.get('final_metrics', {})
        print(f"\n  {name}:")
        print(f"    Params:    {cfg.get('params_M', '?')}M")
        for k, v in fm.items():
            if isinstance(v, (int, float)):
                print(f"    {k:15s}: {v:.4f}")

## Part C: Quantization Experiments

In [ ]:
# === Cell 5: PTQ — Post-Training Quantization ===
# Requires a trained model — use one from VOC or COCO experiments
# If no VOC model exists yet, train a quick one first:

import os
from pathlib import Path

weights = 'experiments/results/ablation-a4-relu6/best.pt'
if not Path(weights).exists():
    # Train a quick model for quantization testing
    print("Training quick model for quantization...")
    os.system(
        "python scripts/train.py --task det --variant quantized "
        "--imgsz 416 --epochs 50 --seed 42 --warmup 3 --name quant-baseline"
    )
    weights = 'experiments/results/quant-baseline/best.pt'

print(f"\nUsing weights: {weights}")

# PTQ
print("\n" + "="*60)
print("  PTQ — Post-Training Quantization")
print("="*60)
os.system(
    f"python scripts/quantize.py --mode ptq "
    f"--weights {weights} --task det --variant quantized "
    f"--imgsz 416 --n-calib 500 --backend qnnpack"
)
print("\n✅ PTQ complete!")

In [ ]:
# === Cell 6: QAT — Quantization-Aware Training ===
print("="*60)
print("  QAT — Quantization-Aware Training")
print("="*60)
os.system(
    f"python scripts/quantize.py --mode qat "
    f"--weights {weights} --task det --variant quantized "
    f"--imgsz 416 --epochs 10 --lr 1e-4 --backend qnnpack"
)
print("\n✅ QAT complete!")

In [ ]:
# === Cell 7: Export INT8 Models ===
print("Exporting INT8 ONNX (PTQ)...")
os.system(
    f"python scripts/quantize.py --mode ptq "
    f"--weights {weights} --task det --variant quantized "
    f"--n-calib 500 --export onnx"
)

# Also export FP32 for size comparison
print("\nExporting FP32 ONNX...")
os.system(
    f"python scripts/export.py --weights {weights} "
    f"--task det --variant quantized --imgsz 416 --formats onnx,torchscript"
)

# Measure sizes
print("\n" + "="*60)
print("  Model Sizes")
print("="*60)
for ext in ['*.onnx', '*.torchscript', '*.pt']:
    for f in sorted(Path('experiments').rglob(ext)):
        print(f"  {f.name}: {f.stat().st_size / 1e6:.2f} MB")

print("\n✅ All exports complete!")

In [ ]:
# === Cell 8: Quantization Comparison Plot ===
import matplotlib.pyplot as plt
import numpy as np

# Collect quantization results
quant_results = []
for f in sorted(Path('experiments').rglob('*.json')):
    if 'ptq' in str(f).lower() or 'qat' in str(f).lower():
        with open(f) as fh:
            data = json.load(fh)
        quant_results.append({'file': f.name, **data})

if quant_results:
    print("Quantization results found:")
    for r in quant_results:
        print(f"  {r['file']}")

# Model size comparison
sizes = {}
for f in sorted(Path('experiments').rglob('*')):
    if f.suffix in ['.onnx', '.pt', '.torchscript']:
        sizes[f.name] = f.stat().st_size / 1e6

if sizes:
    fig, ax = plt.subplots(figsize=(10, 5))
    names = list(sizes.keys())
    vals = list(sizes.values())
    bars = ax.barh(names, vals, color='#2196F3', alpha=0.8, edgecolor='#333')
    ax.set_xlabel('Size (MB)')
    ax.set_title('Model Size Comparison')
    for bar, v in zip(bars, vals):
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2, f'{v:.2f} MB',
                va='center', fontsize=10)
    plt.tight_layout()
    plt.savefig('experiments/results/model_sizes.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("Saved: experiments/results/model_sizes.png")

In [ ]:
!cd experiments && zip -r /content/multitask_quant_results.zip results/seg-* results/pose-* results/quant-* 2>/dev/null
print("📥 Download: /content/multitask_quant_results.zip")